 # Installing Dependencies

In [ ]:
 #!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [111]:
#pip install replicate

In [112]:
from langchain_core.documents import Document

In [113]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

In [114]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [115]:
type(sample_doc)

langchain_core.documents.base.Document

## Ingestion pipeline

In [116]:
# Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt", encoding="utf-8")

In [117]:
document = loader.load()

In [118]:
document

[Document(metadata={'source': 'data/Python.txt'}, page_content='--------OPEN IN BROWSER-------------------\n\nhttps://lwfiles.mycourse.app/62a6cd5e1e9e2fbf212d608d-public/publicFiles/Python.txt')]

In [119]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

In [120]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

## Text to Documents

In [121]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [122]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [123]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [124]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

## Documents to Chunks

In [126]:
# chunks
#!pip install langchain_text_splitters

In [127]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [128]:
chunks = split_docs(all_pdf_documents)

In [129]:
len(chunks)

320

## Embedings

In [130]:
from sentence_transformers import SentenceTransformer

In [131]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [132]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2289.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dimensions= 384


## Vector Store

In [133]:
import chromadb
import uuid

In [134]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [135]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 320


In [136]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 10/10 [00:22<00:00,  2.24s/it]


embeddings shape: (320, 384)
total documents added in vector store= 320
docs in collection: 640


## RAG Implementation

In [137]:
from sklearn.metrics.pairwise import cosine_similarity

In [138]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [139]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [140]:
rag_retriever.retrieve("What is encoder decoder")

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 22.74it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_9b218605-6e6f-4f6b-a25d-05f703f57e7e',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7',
  'metadata': {'title': 'Attention is All you Need',
   'moddate': '2026-03-22T10:14:18+05:30',
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'page_label': '7',
   'published': '2017',
   'source': 'data/pdfs\\reasearch1.pdf',
   'book': 'Advances in Neural Information Processing Systems 30',
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine 

# LLMS with RAG

### GEMNEI

In [36]:
#!pip install langchain-openai

In [142]:
#pip install -U langchain-google-genai

In [143]:
from dotenv import load_dotenv
import os

# Load .env before initializing anything
load_dotenv(".env")

# Optionally set it explicitly for safety
api_key = os.getenv("GEMANAI_API_KEY")
if not api_key:
    raise ValueError("GEMANAI_API_KEY not found in .env")
os.environ["GEMANAI_API_KEY"] = api_key

In [144]:
from dotenv import load_dotenv
import os

# Load .env first
load_dotenv(".env")

# Get the API key
api_key = os.getenv("GEMANAI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found in .env")

# Set environment variable (optional but recommended)
os.environ["GOOGLE_API_KEY"] = api_key

# Import LangChain after environment is set
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    api_key=api_key,
    model="gemini-2.5-flash",   # Use a supported public model
    temperature=0.1,
    max_output_tokens=2048
)

In [161]:
def generate_output(query, retriever, llm, top_k=3):
    """
    Generate an answer using RAG (retrieved documents) + LLM general knowledge.
    """
    # Retrieve top_k relevant documents
    results = retriever.retrieve(query, top_k)
    context = "\n".join([doc["document"] for doc in results]) if results else ""

    # Build prompt
    if context:
        prompt = f"""
        Using the context below and your general knowledge, provide a clear and concise answer to the query.
        Include the full form of any acronym if applicable.

        Context: {context}
        Query: {query}
        """
    else:
        # No context: rely solely on LLM knowledge
        prompt = f"""
        Provide a clear and concise answer to the query using your general knowledge.
        Include the full form of any acronym if applicable.

        Query: {query}
        """

    # Call the LLM
    # Use `generate` or `invoke` depending on your Gemini LangChain setup
    response = llm.invoke(prompt) if hasattr(llm, "invoke") else llm.generate(prompt)
    
    # Return clean output
    return response.content.strip() if hasattr(response, "content") else response

In [172]:
# Example usage
# answer = generate_output("AI /ML Road Map", rag_retriever, llm)
# print(answer)

### GROQ

In [174]:
#!pip install langchain-groq

In [179]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

# Load .env first
load_dotenv(".env")

# Get the API key
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY not found in .env")

# Set environment variable (optional but recommended)
os.environ["GROQ_API_KEY"] = api_key

llm = ChatGroq(
    api_key = api_key,
    model_name="qwen/qwen3-32b",
    temperature=0.7,
    max_tokens=1024
)

In [182]:
def generate_output(query, retriever, llm, top_k=3):
    """
    Generate an answer using RAG (retrieved documents) + LLM general knowledge.
    """
    # Retrieve top_k relevant documents
    results = retriever.retrieve(query, top_k)
    context = "\n".join([doc["document"] for doc in results]) if results else ""

    # Build prompt
    if context:
        prompt = f"""
        Using the context below and your general knowledge, provide a clear and concise answer to the query.
        Include the full form of any acronym if applicable.

        Context: {context}
        Query: {query}
        """
    else:
        prompt = f"""
        Provide a clear and concise answer to the query using your general knowledge.
        Include the full form of any acronym if applicable.

        Query: {query}
        """

    # Call the LLM
    if hasattr(llm, "invoke"):
        response = llm.invoke(prompt)           # returns AIMessage
        output = response.content.strip()       # get the string from AIMessage
    else:
        response = llm.generate(prompt)
        output = response.strip() if isinstance(response, str) else response

    return output

In [183]:
answer = generate_output("AI /ML Road Map", rag_retriever, llm)

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.63it/s]


embeddings shape: (1, 384)
retrieved 0 documents


In [184]:
print(answer)

<think>
Okay, the user is asking for an AI/ML Road Map. Let me start by breaking down what they might need. They probably want a structured path to learn AI and Machine Learning, from the basics to more advanced topics. First, I should outline the main areas they need to cover.

Starting with the foundation, mathematics is crucial. Linear algebra, calculus, probability, and statistics are the base. Without these, understanding algorithms can be tough. Then programming—Python is the go-to language here. Maybe mention some libraries like NumPy, Pandas, and Scikit-learn.

Next, moving on to core machine learning concepts. They should learn about different types of learning: supervised, unsupervised, reinforcement. Important algorithms like linear regression, decision trees, SVMs, etc. Also, model evaluation metrics are important for assessing performance.

Then deep learning comes into play. Frameworks like TensorFlow and PyTorch are essential. They need to understand neural networks basi

## Deepseek 

In [186]:
#pip install -U langchain-deepseek

In [187]:
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
import os

# Load .env first
load_dotenv(".env")

# Get the API key
api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    raise ValueError("DEEPSEEK_API_KEY not found in .env")

# Set environment variable (optional but recommended)
os.environ["DEEPSEEK_API_KEY"] = api_key

model = ChatDeepSeek(
    api_key = api_key,
    model="deepseek-cha",
    temperature=0.7,
    max_tokens=1024
)

In [188]:
def generate_output(query, retriever, llm, top_k=3):
    """
    Generate an answer using RAG (retrieved documents) + LLM general knowledge.
    Compatible with ChatDeepSeek.
    """
    # Retrieve top_k relevant documents
    results = retriever.retrieve(query, top_k)
    context = "\n".join([doc["document"] for doc in results]) if results else ""

    # Build prompt
    prompt = f"""
    Using the context below and your general knowledge, provide a clear and concise answer.
    Include full forms of any acronyms.

    Context: {context if context else 'No context available.'}
    Query: {query}
    """

    # Call the LLM
    if hasattr(llm, "invoke"):
        response = llm.invoke(prompt)
        return response.content.strip()
    else:
        response = llm.generate(prompt)
        return response.strip() if isinstance(response, str) else response

In [191]:
#answer = generate_output("What is RAG?", rag_retriever, model)
